In [1]:
import os
import pandas as pd
import json
Dir_project = '/work/desai-lab/xuanyang/Project/dataset/Nastase/narratives'
Dir_gentle = '/work/desai-lab/xuanyang/Project/dataset/Nastase/narratives/stimuli/gentle'

#### Load fMRI data needs to be run

In [2]:
# fmri Data
df_scans_filter = pd.read_csv('/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI/scripts/fMRI/master_subject_10stories_highacc.csv')
df_scans_filter

,subID,task,age,sex,condition,comprehension,transcript,label
0,sub-023,prettymouth,28,F,affair,0.889,prettymouth,prettymouth
1,sub-023,milkyway,28,F,vodka,1.000,milkywayvodka,milkywayvodka
2,sub-030,prettymouth,21,F,paranoia,0.963,prettymouth,prettymouth
3,sub-030,milkyway,21,F,vodka,0.893,milkywayvodka,milkywayvodka
4,sub-032,prettymouth,22,M,affair,0.963,prettymouth,prettymouth
...,...,...,...,...,...,...,...,...
208,sub-314,piemanpni,25,F,NaN,0.767,piemanpni,piemanpni
209,sub-314,bronx,25,F,NaN,0.780,bronx,bronx
210,sub-314,forgot,25,F,NaN,0.780,forgot,forgot
211,sub-314,black,25,F,NaN,0.920,black,black


In [3]:
df_master_blankTR = pd.DataFrame({'transcript':df_scans_filter.transcript.unique()})
df_master_blankTR = df_master_blankTR.sort_values('transcript').reset_index(drop=True)
df_master_blankTR['blankTR'] = [0, 8, 8, 8, 0, 0, 8, 0, 3, 3]
df_master_blankTR

,transcript,blankTR
0,21styear,0
1,black,8
2,bronx,8
3,forgot,8
4,milkywayoriginal,0
5,milkywayvodka,0
6,piemanpni,8
7,prettymouth,0
8,shapessocial,3
9,slumlordreach,3


In [4]:
import nltk
dfs_timestamps = []
for transcript in df_scans_filter.transcript.unique():
    df_timestamps = pd.read_csv(os.path.join(Dir_gentle,transcript,'align.csv'),header=None)
    df_timestamps.columns = ['Word','word_gentle','onset','offset']
    df = pd.DataFrame(nltk.pos_tag(df_timestamps['Word'].str.lower()),columns = ['Word','PoS'])
    df_timestamps['PoS'] = df['PoS']
    df_timestamps['transcript'] = transcript
    dfs_timestamps.append(df_timestamps)
dfs_timestamps = pd.concat(dfs_timestamps,ignore_index=True)

# lemmatize nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
for word in dfs_timestamps['Word'].unique():
    dfs_timestamps.loc[dfs_timestamps['Word']==word,'lemma'] = lemmatizer.lemmatize(word.lower())
dfs_timestamps['word'] = dfs_timestamps['Word'].str.lower()
dfs_timestamps

,Word,word_gentle,onset,offset,PoS,transcript,lemma,word
0,When,when,1.340000,1.500000,WRB,prettymouth,when,when
1,the,the,21.150000,21.250000,DT,prettymouth,the,the
2,phone,phone,21.250000,21.560000,NN,prettymouth,phone,phone
3,rang,rang,21.560000,21.990000,VBD,prettymouth,rang,rang
4,the,the,22.070000,22.190000,DT,prettymouth,the,the
...,...,...,...,...,...,...,...,...
24653,C,c,818.280000,818.710000,NN,forgot,c,c
24654,D,d,819.809999,820.159999,NN,forgot,d,d
24655,E,e,821.540000,821.880000,NN,forgot,e,e
24656,F,f,822.969999,823.429999,NN,forgot,f,f


#### shift the timings to match the fmri time series

In [8]:
TR = 1.5
Dir_codes= '/work/desai-lab/xuanyang/Project/Semantic/dissemination/github/DiscoFMRI/scripts/fMRI/timings/'
os.makedirs(Dir_codes,exist_ok=True)
dfs_timestamps['blankTR'] = 0
for transcript in dfs_timestamps.transcript.unique():
    blankTR = df_master_blankTR.loc[df_master_blankTR.transcript==transcript,'blankTR'].values[0]
#     dfs_timestamps.loc[dfs_timestamps.transcript == transcript,['onset_addblankTR','offset_addblankTR']] = dfs_timestamps.loc[dfs_timestamps.transcript == transcript,['onset','offset']] + [blankTR*TR,blankTR*TR]
    dfs_timestamps.loc[dfs_timestamps.transcript==transcript,'blankTR'] = blankTR
dfs_timestamps['onset_addblankTR'] = dfs_timestamps['onset'] + dfs_timestamps['blankTR'] *TR
dfs_timestamps['offset_addblankTR'] = dfs_timestamps['offset'] + dfs_timestamps['blankTR'] *TR
dfs_timestamps.to_csv(os.path.join(Dir_codes,'df_timings_10transcripts.csv'),index=False)
dfs_timestamps

,Word,word_gentle,onset,offset,PoS,transcript,lemma,word,blankTR,onset_addblankTR,offset_addblankTR
0,When,when,1.340000,1.500000,WRB,prettymouth,when,when,0,1.340000,1.500000
1,the,the,21.150000,21.250000,DT,prettymouth,the,the,0,21.150000,21.250000
2,phone,phone,21.250000,21.560000,NN,prettymouth,phone,phone,0,21.250000,21.560000
3,rang,rang,21.560000,21.990000,VBD,prettymouth,rang,rang,0,21.560000,21.990000
4,the,the,22.070000,22.190000,DT,prettymouth,the,the,0,22.070000,22.190000
...,...,...,...,...,...,...,...,...,...,...,...
24653,C,c,818.280000,818.710000,NN,forgot,c,c,8,830.280000,830.710000
24654,D,d,819.809999,820.159999,NN,forgot,d,d,8,831.809999,832.159999
24655,E,e,821.540000,821.880000,NN,forgot,e,e,8,833.540000,833.880000
24656,F,f,822.969999,823.429999,NN,forgot,f,f,8,834.969999,835.429999
